In [31]:
import sys
from pathlib import Path
import json

# Add the repo root to sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from thesis_rec import thesis_rec

In [32]:
def load_all_records():
    base_path = Path("..") / "thesis_data"
    all_records = []

    for species_dir in base_path.glob("*/madb"):
        species = species_dir.parent.name  # e.g., "human_chimp"
        for json_file in species_dir.glob("ENSG*.json"):
            try:
                with open(json_file) as f:
                    outer = json.load(f)
                    rich_dict_str = outer.get("data")
                    if rich_dict_str is None:
                        raise ValueError(f"No 'data' field in {json_file}")
                    rich_dict = json.loads(rich_dict_str)
                    record = thesis_rec.from_rich_dict(rich_dict)
                    record.species = species  # add as dynamic attribute
                    all_records.append(record)
            except Exception as e:
                print(f"Failed to load {json_file}: {e}")
    
    return all_records


In [33]:
def extract_lengths(records):
    sw_lengths = []
    braid_lengths = []
    ids = []
    species_labels = []
    used_kmers = []

    for rec in records:
        braid_data = rec.madb_longest_braid_length
        sw_len = rec.madb_ungapped_smith_waterman_length

        if sw_len is None or not isinstance(braid_data, dict):
            continue

        # Filter only valid k-mer entries with non-null braid lengths
        valid_braids = {int(k): v for k, v in braid_data.items() if v is not None}

        if not valid_braids:
            continue

        # Choose k with longest braid (you can change this logic)
        selected_k = max(valid_braids, key=valid_braids.get)
        braid_len = valid_braids[selected_k]

        sw_lengths.append(sw_len)
        braid_lengths.append(braid_len)
        ids.append(rec.unique_id)
        species_labels.append(getattr(rec, "species", "unknown"))
        used_kmers.append(selected_k)

    return sw_lengths, braid_lengths, ids, species_labels, used_kmers


In [43]:
import plotly.express as px
import pandas as pd
from pathlib import Path

# Load data
records = load_all_records()
sw_lengths, braid_lengths, ids, species_labels, used_kmers = extract_lengths(records)

df = pd.DataFrame({
    "SW Length": sw_lengths,
    "Longest Braid Length": braid_lengths,
    "ID": ids,
    "Species": species_labels,
    "K-mer size": used_kmers
})

# Determine axis max value for symmetric plot
max_val = max(df["SW Length"].max(), df["Longest Braid Length"].max()) * 1.05  # add 5% margin

# Create plot
fig = px.scatter(
    df,
    x="SW Length",
    y="Longest Braid Length",
    hover_name="ID",
    hover_data=["Species", "K-mer size"],
)

# Remove legend and set single color
fig.update_traces(marker=dict(color='black'), showlegend=False)

# Add parity line (y = x)
fig.add_shape(
    type="line",
    x0=0, y0=0, x1=max_val, y1=max_val,
    line=dict(dash="dot", color="gray")
)

fig.update_layout(
    xaxis=dict(
        range=[0, max_val],
        title="Length (Smith-Waterman)",
        constrain='domain',
        fixedrange=True
    ),
    yaxis=dict(
        range=[0, max_val],
        title="Length (madb)",
        scaleanchor="x",
        scaleratio=1,
        constrain='domain',
        fixedrange=True
    ),
    width=400,
    height=400,
    margin=dict(l=60, r=60, t=50, b=50),
)

figures_dir = Path("..") / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

fig.write_html(figures_dir / "sw_vs_braid.html")
fig.write_image(figures_dir / "sw_vs_braid.png", width=800, height=800)

fig.show()
